In [1]:
!pip install transformers torch scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# =========================
# 1. Load saved splits
# =========================
train_df = pd.read_parquet("data/splits/train.parquet")
val_df = pd.read_parquet("data/splits/val.parquet")
test_df = pd.read_parquet("data/splits/test.parquet")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# =========================
# 2. Check label format
# human = 0, ai = 1
# =========================
print("\nTest label distribution:")
print(test_df["label"].value_counts().sort_index())

# =========================
# 3. Device setting
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing device:", device)

# =========================
# 4. Load an existing public AI-text detection model
# Replace this model name if needed
# =========================
MODEL_NAME = "Hello-SimpleAI/chatgpt-detector-roberta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("\nLoaded model:", MODEL_NAME)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

# =========================
# 5. Prepare test data
# =========================
test_texts = test_df["text"].astype(str).tolist()
y_true = test_df["label"].astype(int).tolist()

# =========================
# 6. Prediction function
# =========================
def predict_in_batches(texts, tokenizer, model, device, batch_size=16, max_length=512):
    all_preds = []
    all_probs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return np.array(all_preds), np.array(all_probs)

# =========================
# 7. Run prediction
# =========================
y_pred, y_prob = predict_in_batches(
    test_texts,
    tokenizer,
    model,
    device,
    batch_size=16,
    max_length=512
)

# =========================
# 8. IMPORTANT:
# Check whether model label order matches your dataset
# Your dataset: human = 0, ai = 1
# If model uses the reverse order, flip predictions
# =========================
print("\nUnique predicted labels before adjustment:", np.unique(y_pred))

# Example manual check:
# If model.config.id2label is something like {0: 'LABEL_0', 1: 'LABEL_1'},
# you may need to inspect some examples manually.
# Only use the following line if you confirm the model output is reversed:
# y_pred = 1 - y_pred
# y_prob = y_prob[:, ::-1]

# =========================
# 9. Evaluation
# =========================
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="binary")
recall = recall_score(y_true, y_pred, average="binary")
f1 = f1_score(y_true, y_pred, average="binary")

print("\nExisting Model Results")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

# =========================
# 10. Save predictions
# =========================
results_df = test_df.copy()
results_df["pred_existing_model"] = y_pred
results_df["prob_class_0"] = y_prob[:, 0]
results_df["prob_class_1"] = y_prob[:, 1]

results_df.to_csv("data/splits/existing_model_test_predictions.csv", index=False)
print("\nSaved predictions to: data/splits/existing_model_test_predictions.csv")

d:\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: 'data/splits/train.parquet'